# Unify BIDS Events Trial Type

This notebook standardizes Alice BIDS `events.tsv` files so stimulus markers use one consistent format.

Problem examples currently seen:

```text
Stimulus/1
Stimulus/S  1
Stimulus/S 12
```

Target format:

```text
trial_type = Stimulus/1 ... Stimulus/12
stimulus_id = 1 ... 12
raw_trial_type = original trial_type before normalization
```

Default mode is dry run. Set `DRY_RUN = False` only after reviewing the report.

In [1]:
from pathlib import Path
from datetime import datetime
import re
import shutil

import pandas as pd

BIDS_ROOT = Path('/Users/yanyuwoo/Data/bids')
ANALYSIS_ROOT = Path('/Users/yanyuwoo/Data/Alice Comprehension')
QC_DIR = ANALYSIS_ROOT / 'qc'
BACKUP_ROOT = ANALYSIS_ROOT / 'intermediate' / 'bids_events_backups'

# Keep True until the dry-run report looks right.
DRY_RUN = False

QC_DIR.mkdir(parents=True, exist_ok=True)
BACKUP_ROOT.mkdir(parents=True, exist_ok=True)

print(f'BIDS root: {BIDS_ROOT}')
print(f'DRY_RUN: {DRY_RUN}')

BIDS root: /Users/yanyuwoo/Data/bids
DRY_RUN: False


## Parser

The parser extracts the final stimulus number from both known formats and rejects anything outside `1..12`.

In [5]:
STIMULUS_RE = re.compile(r'^Stimulus/(?:S\s*)?(?P<stimulus_id>\d+)$')


def parse_stimulus_id(trial_type):
    if not isinstance(trial_type, str):
        return None
    match = STIMULUS_RE.match(trial_type.strip())
    if match is None:
        return None
    stimulus_id = int(match.group('stimulus_id'))
    if not 1 <= stimulus_id <= 12:
        return None
    return stimulus_id


assert parse_stimulus_id('Stimulus/1') == 1
assert parse_stimulus_id('Stimulus/12') == 12
assert parse_stimulus_id('Stimulus/S  1') == 1
assert parse_stimulus_id('Stimulus/S 12') == 12
print('Parser sanity check passed.')

Parser sanity check passed.


## Inspect all events files

In [6]:
events_paths = sorted(
    BIDS_ROOT.glob('sub-*/eeg/sub-*_task-alice_events.tsv'),
    key=lambda p: int(p.parts[-3].split('-')[1]),
)

rows = []
for path in events_paths:
    subject = path.parts[-3]
    df = pd.read_csv(path, sep='\t', encoding='utf-8-sig')
    parsed = df['trial_type'].map(parse_stimulus_id)
    rows.append({
        'subject': subject,
        'path': str(path),
        'n_rows': len(df),
        'n_stimulus_rows': int(parsed.notna().sum()),
        'stimulus_ids': ';'.join(map(str, sorted(parsed.dropna().astype(int).tolist()))),
        'missing_stimulus_ids': ';'.join(map(str, sorted(set(range(1, 13)) - set(parsed.dropna().astype(int).tolist())))),
        'original_trial_types': '; '.join(df['trial_type'].astype(str).tolist()),
        'needs_change': bool((df['trial_type'].astype(str) != parsed.map(lambda x: f'Stimulus/{int(x)}' if pd.notna(x) else None).astype(str)).any()),
        'unparsed_trial_types': '; '.join(df.loc[parsed.isna(), 'trial_type'].astype(str).tolist()),
    })

report = pd.DataFrame(rows)
report_path = QC_DIR / 'bids_events_trial_type_unify_dry_run.csv'
report.to_csv(report_path, index=False)
report[['subject', 'n_rows', 'n_stimulus_rows', 'missing_stimulus_ids', 'needs_change', 'unparsed_trial_types']]

,subject,n_rows,n_stimulus_rows,missing_stimulus_ids,needs_change,unparsed_trial_types
0,sub-01,12,12,,False,
1,sub-02,11,11,1,False,
2,sub-03,12,12,,False,
3,sub-04,12,12,,False,
4,sub-05,12,12,,False,
5,sub-06,12,12,,False,
6,sub-07,12,12,,False,
7,sub-08,12,12,,False,
8,sub-09,12,12,,False,
9,sub-10,12,12,,False,


## Apply normalization

This cell writes only when `DRY_RUN = False`.

For each file:

1. Copy the original file into a timestamped backup directory.
2. Add `raw_trial_type` if absent.
3. Add/update `stimulus_id`.
4. Replace parseable stimulus rows in `trial_type` with `Stimulus/N`.
5. Save the updated `events.tsv`.

In [7]:
timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
backup_dir = BACKUP_ROOT / timestamp

apply_rows = []

for path in events_paths:
    subject = path.parts[-3]
    df = pd.read_csv(path, sep='\t', encoding='utf-8-sig')
    original_df = df.copy()

    parsed = df['trial_type'].map(parse_stimulus_id)
    parseable = parsed.notna()

    if 'raw_trial_type' not in df.columns:
        df['raw_trial_type'] = df['trial_type']

    df['stimulus_id'] = pd.NA
    df.loc[parseable, 'stimulus_id'] = parsed[parseable].astype(int)
    df.loc[parseable, 'trial_type'] = parsed[parseable].astype(int).map(lambda x: f'Stimulus/{x}')

    changed = not df.equals(original_df)

    if changed and not DRY_RUN:
        backup_dir.mkdir(parents=True, exist_ok=True)
        backup_path = backup_dir / path.name
        shutil.copy2(path, backup_path)
        df.to_csv(path, sep='\t', index=False)
    else:
        backup_path = None

    apply_rows.append({
        'subject': subject,
        'path': str(path),
        'changed': changed,
        'written': bool(changed and not DRY_RUN),
        'backup_path': str(backup_path) if backup_path else '',
        'n_stimulus_rows': int(parseable.sum()),
        'missing_stimulus_ids': ';'.join(map(str, sorted(set(range(1, 13)) - set(parsed.dropna().astype(int).tolist())))),
    })

apply_report = pd.DataFrame(apply_rows)
suffix = 'dry_run' if DRY_RUN else 'applied'
apply_report_path = QC_DIR / f'bids_events_trial_type_unify_{suffix}.csv'
apply_report.to_csv(apply_report_path, index=False)

print(f'Wrote report: {apply_report_path}')
if not DRY_RUN:
    print(f'Backups: {backup_dir}')

apply_report

Wrote report: /Users/yanyuwoo/Data/Alice Comprehension/qc/bids_events_trial_type_unify_applied.csv
Backups: /Users/yanyuwoo/Data/Alice Comprehension/intermediate/bids_events_backups/20260630-165654


,subject,path,changed,written,backup_path,n_stimulus_rows,missing_stimulus_ids
0,sub-01,/Users/yanyuwoo/Data/bids/sub-01/eeg/sub-01_ta...,True,True,/Users/yanyuwoo/Data/Alice Comprehension/inter...,12,
1,sub-02,/Users/yanyuwoo/Data/bids/sub-02/eeg/sub-02_ta...,True,True,/Users/yanyuwoo/Data/Alice Comprehension/inter...,11,1
2,sub-03,/Users/yanyuwoo/Data/bids/sub-03/eeg/sub-03_ta...,True,True,/Users/yanyuwoo/Data/Alice Comprehension/inter...,12,
3,sub-04,/Users/yanyuwoo/Data/bids/sub-04/eeg/sub-04_ta...,True,True,/Users/yanyuwoo/Data/Alice Comprehension/inter...,12,
4,sub-05,/Users/yanyuwoo/Data/bids/sub-05/eeg/sub-05_ta...,True,True,/Users/yanyuwoo/Data/Alice Comprehension/inter...,12,
5,sub-06,/Users/yanyuwoo/Data/bids/sub-06/eeg/sub-06_ta...,True,True,/Users/yanyuwoo/Data/Alice Comprehension/inter...,12,
6,sub-07,/Users/yanyuwoo/Data/bids/sub-07/eeg/sub-07_ta...,True,True,/Users/yanyuwoo/Data/Alice Comprehension/inter...,12,
7,sub-08,/Users/yanyuwoo/Data/bids/sub-08/eeg/sub-08_ta...,True,True,/Users/yanyuwoo/Data/Alice Comprehension/inter...,12,
8,sub-09,/Users/yanyuwoo/Data/bids/sub-09/eeg/sub-09_ta...,True,True,/Users/yanyuwoo/Data/Alice Comprehension/inter...,12,
9,sub-10,/Users/yanyuwoo/Data/bids/sub-10/eeg/sub-10_ta...,True,True,/Users/yanyuwoo/Data/Alice Comprehension/inter...,12,
